# Baseline LSTM — Sequential Model

**Author:** Minho  
**Project:** Risk-Adjusted Portfolio Optimization  
**Model:** LSTM (sequential baseline)  
**Dataset:** `shared_set_2` (Growth/Tech/Innovation universe)  
**MLflow run name:** `Minho_Baseline_LSTM`

---

### What this notebook does

This is the simplest working LSTM baseline for the team's shared research framework.
It is intentionally kept minimal and readable — no tricks, no optimizations.

Steps:
1. Load shared dataset via `portfolio_toolkit`
2. Build shared toolkit features + 1 custom feature (`price_accel`)
3. Build per-ticker rolling sequences for the LSTM
4. Train a 2-layer LSTM to predict 5-day forward returns
5. Emit a standardized prediction table
6. Convert predictions → portfolio weights
7. Run the shared backtest
8. Log everything to MLflow as `Minho_Baseline_LSTM`

### How to run

```
cd <repo-root>
source venv312/bin/activate
jupyter notebook MODELS/Minho/baseline_lstm.ipynb
```

Run all cells top to bottom. No hidden state — every cell is self-contained.

## 0. Bootstrap — locate repo root

In [ ]:
import os
import sys
from pathlib import Path

# --- If auto-detection fails, set this manually ---
# repo_root = Path('/Users/minhochoi/Portfolio-Optimization-Lib')
# --------------------------------------------------

def _is_repo_root(p: Path) -> bool:
    return (p / 'pyproject.toml').exists() and (p / 'src' / 'portfolio_toolkit').exists()

if 'repo_root' not in dir() or not _is_repo_root(Path(repo_root)):
    _candidates = [Path.cwd(), *Path.cwd().parents]
    _found = next((p for p in _candidates if _is_repo_root(p)), None)
    if _found is None:
        raise RuntimeError(
            'Cannot find repo root. Uncomment and set repo_root manually at the top of this cell.'
        )
    repo_root = _found

repo_root = Path(repo_root).resolve()
os.chdir(repo_root)

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('repo_root:', repo_root)
print('python:   ', sys.executable)

## 1. Imports

In [ ]:
import random
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_return_target,
    slice_split,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_rank_long_only,
    write_backtest_artifacts,
)

warnings.filterwarnings('ignore')
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Config

All hyperparameters in one place.

In [ ]:
DATASET_NAME  = 'shared_set_2'
HORIZON       = 5          # forecast horizon in trading days
RUN_NAME      = 'Minho_Baseline_LSTM'

SEQ_LEN       = 20         # look-back window (trading days)
HIDDEN_SIZE   = 64
NUM_LAYERS    = 2
DROPOUT       = 0.2
LR            = 1e-3
EPOCHS        = 30
BATCH_SIZE    = 512
SEED          = 42

artifact_dir  = repo_root / 'outputs' / RUN_NAME
artifact_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)
print(f'Dataset : {DATASET_NAME}  ({len(spec.tickers)} tickers)')
print(f'Train   : {spec.train_start} → {spec.train_end}')
print(f'Val     : {spec.val_start}   → {spec.val_end}')
print(f'Test    : {spec.test_start}  → {spec.test_end}')

## 3. Load Prices

In [ ]:
prices = load_prices(DATASET_NAME, repo_root=repo_root)
print('Shape     :', prices.shape)
print('Date range:', prices['date'].min().date(), '→', prices['date'].max().date())
prices.head(3)

## 4. Features — Shared Toolkit + 1 Custom

**Custom feature: `price_accel`**  
Defined as `return_5d − return_20d`. Captures whether short-term momentum is
accelerating (positive) or decelerating (negative) relative to the medium-term trend.
Particularly informative for an LSTM because the model can learn how acceleration
evolves across the look-back window.

In [ ]:
BASE_FEATURES = [
    'return_1d', 'return_5d', 'return_20d',
    'vol_5d', 'vol_20d',
    'momentum_5d', 'momentum_20d', 'momentum_60d',
    'rsi_14', 'macd_hist', 'bollinger_z_20d',
    'beta_20d_spy', 'excess_return_5d_vs_spy',
    'volume_zscore_20d', 'price_to_sma_20d', 'atr_14',
]

feature_frame = build_features(prices, feature_names=BASE_FEATURES)

# --- Custom feature: price acceleration ---
panel = prices.sort_values(['ticker', 'date'])
ret5  = panel.groupby('ticker')['adj_close'].pct_change(5)
ret20 = panel.groupby('ticker')['adj_close'].pct_change(20)

custom_df = panel[['date', 'ticker']].copy()
custom_df['price_accel'] = (ret5 - ret20).values

frame = feature_frame.merge(custom_df, on=['date', 'ticker'], how='left')
ALL_FEATURES = BASE_FEATURES + ['price_accel']

print(f'Features: {len(ALL_FEATURES)}  ({len(BASE_FEATURES)} shared + 1 custom)')
frame[['date', 'ticker', 'price_accel']].dropna().head(3)

## 5. Targets + Train / Val / Test Split

In [ ]:
TARGET_COL = f'forward_return_{HORIZON}d'

targets      = make_forward_return_target(prices, horizon=HORIZON)
target_frame = frame.merge(targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

train = slice_split(target_frame, DATASET_NAME, 'train', repo_root=repo_root)
val   = slice_split(target_frame, DATASET_NAME, 'val',   repo_root=repo_root)
test  = slice_split(target_frame, DATASET_NAME, 'test',  repo_root=repo_root)

# Normalize using train statistics only (no leakage)
train_mean = train[ALL_FEATURES].mean()
train_std  = train[ALL_FEATURES].std(ddof=0).replace(0.0, 1.0)

def scale(df):
    return ((df[ALL_FEATURES] - train_mean) / train_std).to_numpy(dtype=np.float32)

X_train, X_val, X_test = scale(train), scale(val), scale(test)
y_train = train[TARGET_COL].to_numpy(dtype=np.float32)
y_val   = val[TARGET_COL].to_numpy(dtype=np.float32)

print(f'Train : {len(train):>6,} rows    X_train : {X_train.shape}')
print(f'Val   : {len(val):>6,} rows    X_val   : {X_val.shape}')
print(f'Test  : {len(test):>6,} rows    X_test  : {X_test.shape}')

## 6. Build Per-Ticker Sequences

LSTMs consume `(batch, seq_len, n_features)` tensors. We build overlapping
windows of length `SEQ_LEN` **within each ticker** so sequences never span
ticker boundaries. The label is the target at the last timestep of each window.

In [ ]:
def make_sequences(
    df: pd.DataFrame,
    X: np.ndarray,
    y: np.ndarray,
    seq_len: int,
):
    """Return (X_seq, y_seq, meta) where meta is list of (date, ticker)."""
    Xs, ys, meta = [], [], []
    df = df.reset_index(drop=True)
    for ticker, grp in df.groupby('ticker', sort=False):
        idx   = grp.index.tolist()
        dates = grp['date'].tolist()
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx)):
            rows = idx[end - seq_len + 1 : end + 1]
            Xs.append(X[rows])
            ys.append(y[idx[end]])
            meta.append((dates[end], ticker))
    return (
        np.stack(Xs).astype(np.float32),
        np.array(ys, dtype=np.float32),
        meta,
    )

print('Building sequences ...')
X_tr_seq, y_tr_seq, _          = make_sequences(train, X_train, y_train, SEQ_LEN)
X_va_seq, y_va_seq, _          = make_sequences(val,   X_val,   y_val,   SEQ_LEN)
X_te_seq, y_te_seq, meta_test  = make_sequences(test,  X_test,  test[TARGET_COL].to_numpy(np.float32), SEQ_LEN)

print(f'Train sequences : {X_tr_seq.shape}')   # (N, SEQ_LEN, F)
print(f'Val sequences   : {X_va_seq.shape}')
print(f'Test sequences  : {X_te_seq.shape}')

## 7. LSTM Model

A minimal 2-layer stacked LSTM. Input → LSTM → last hidden state → linear head → scalar return prediction.
Causal (not bidirectional). Gradient clipping applied during training.

In [ ]:
class BaselineLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)       # h_n: (num_layers, batch, hidden)
        return self.head(h_n[-1]).squeeze(-1)  # scalar per sequence


def train_lstm(model, X_tr, y_tr, X_va, y_va, epochs, batch_size, lr, device):
    """Train and return loss history DataFrame."""
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    loader    = DataLoader(
        TensorDataset(
            torch.as_tensor(X_tr, dtype=torch.float32),
            torch.as_tensor(y_tr, dtype=torch.float32),
        ),
        batch_size=batch_size, shuffle=True,
    )
    history = []
    for epoch in range(epochs):
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            Xt = torch.as_tensor(X_tr, dtype=torch.float32, device=device)
            tr_loss = float(loss_fn(model(Xt), torch.as_tensor(y_tr, device=device)))
            Xv = torch.as_tensor(X_va, dtype=torch.float32, device=device)
            va_loss = float(loss_fn(model(Xv), torch.as_tensor(y_va, device=device)))

        history.append({'epoch': epoch + 1, 'train_loss': tr_loss, 'val_loss': va_loss})
        if (epoch + 1) % 5 == 0:
            print(f'  epoch {epoch+1:3d}/{epochs}  train={tr_loss:.6f}  val={va_loss:.6f}')

    return pd.DataFrame(history)


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print(f'Input size: {X_tr_seq.shape[2]}')

## 8. Train

In [ ]:
model = BaselineLSTM(
    input_size  = X_tr_seq.shape[2],
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT,
)

print(f'Training on {X_tr_seq.shape[0]:,} sequences for {EPOCHS} epochs ...')
history = train_lstm(
    model, X_tr_seq, y_tr_seq, X_va_seq, y_va_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE,
)

best_val = history['val_loss'].min()
print(f'\nBest val MSE: {best_val:.8f}')
history.tail()

## 9. Predictions

In [ ]:
model.eval()
with torch.no_grad():
    raw = model(
        torch.as_tensor(X_te_seq, dtype=torch.float32, device=DEVICE)
    ).cpu().numpy()

pred_dates, pred_tickers = zip(*meta_test)
predictions = pd.DataFrame({
    'date'            : pd.to_datetime(list(pred_dates)),
    'ticker'          : list(pred_tickers),
    'horizon'         : HORIZON,
    'expected_return' : raw.astype(float),
})
predictions = predictions.drop_duplicates(['date', 'ticker', 'horizon'], keep='last')
predictions = validate_prediction_frame(
    predictions, dataset_name=DATASET_NAME, repo_root=repo_root
)

print('Predictions shape:', predictions.shape)
print('Date range:', predictions['date'].min().date(), '→', predictions['date'].max().date())
predictions.head()

## 10. Portfolio Weights

In [ ]:
portfolio = weights_from_predictions_rank_long_only(
    predictions,
    score_column  = 'expected_return',
    dataset_name  = DATASET_NAME,
    strategy_name = RUN_NAME,
)

validated_weights = validate_weights_frame(
    portfolio.weights, dataset_name=DATASET_NAME, repo_root=repo_root
)

print('Weights shape:', validated_weights.shape)
print('Row sums (all should be 1.0):')
print(validated_weights.sum(axis=1).describe())

## 11. Backtest

In [ ]:
result = backtest_weights(
    DATASET_NAME, portfolio, benchmark='SPY', repo_root=repo_root
)

print('Backtest metrics:')
for k, v in sorted(result.metrics.items()):
    print(f'  {k:<35s}: {v:.6f}')

## 12. Write Artifacts (QuantStats report + parquet files)

In [ ]:
artifact_paths = write_backtest_artifacts(result, artifact_dir)

for key, path in artifact_paths.items():
    exists = '✓' if Path(path).exists() else '✗'
    print(f'  {exists} {key:<20s}: {path}')

## 13. Log to MLflow as `Minho_Baseline_LSTM`

In [ ]:
import mlflow

mlflow_layout = init_mlflow(repo_root)
print('Tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name     = RUN_NAME,
    dataset_name = DATASET_NAME,
    tags={
        'author'       : 'Minho',
        'model_family' : 'sequential_lstm',
        'workflow'     : 'baseline_lstm',
        'horizon'      : str(HORIZON),
        'project'      : 'risk_adjusted_portfolio_optimization',
    },
    repo_root=repo_root,
):
    mlflow.log_params({
        'run_name'      : RUN_NAME,
        'dataset'       : DATASET_NAME,
        'horizon'       : HORIZON,
        'seq_len'       : SEQ_LEN,
        'hidden_size'   : HIDDEN_SIZE,
        'num_layers'    : NUM_LAYERS,
        'dropout'       : DROPOUT,
        'lr'            : LR,
        'epochs'        : EPOCHS,
        'batch_size'    : BATCH_SIZE,
        'n_features'    : len(ALL_FEATURES),
        'custom_features': 'price_accel',
        'portfolio_builder': 'rank_long_only',
        'best_val_mse'  : round(float(best_val), 8),
    })
    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

print(f'MLflow run "{RUN_NAME}" logged successfully.')

## 14. Final Checks

In [ ]:
from IPython.display import display

# Sanity assertions
assert {'total_return', 'annual_return', 'sharpe', 'max_drawdown'}.issubset(result.metrics)
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths['quantstats_report']).exists()
assert {'date', 'ticker', 'horizon', 'expected_return'}.issubset(predictions.columns)
assert 'price_accel' in frame.columns

print('All checks passed. Notebook ran end to end successfully.')
print()
print('Key metrics:')
for k in ['annual_return', 'sharpe', 'max_drawdown', 'average_turnover']:
    print(f'  {k:<20s}: {result.metrics[k]:.4f}')

print()
display(result.nav.tail(3).to_frame('nav'))